# Kubernetes Proactive Autoscaler - GRU Workload Prediction Pipeline
This notebook implements the model training and export pipeline described in:
**"Toward Optimal Load Prediction and Customizable Autoscaling Scheme for Kubernetes"** (MDPI Mathematics 2023).

### Pipeline Overview:
1. **Synthetic / Cluster Data Generation**: Simulates 300s/10s interval CPU & memory traces with diurnal cycles and burst traffic.
2. **Data Preprocessing**: Min-Max scaling and 24-step sliding window sequences `(N, 24, 1)`.
3. **GRU Model Training**: 50 hidden units, ReLU activation, Adam optimizer, MSE loss.
4. **Baseline Comparison**: Benchmark vs LSTM, BiLSTM, and ARIMA.
5. **ONNX Export**: Exports lightweight `GRU_Model_24.onnx` for deployment inside the Custom Pod Autoscaler container.

In [ ]:
# 1. Install dependencies (if running on Colab)
!pip install -q onnx onnxruntime scikit-learn pandas matplotlib statsmodels

In [ ]:
# 2. Imports and Environment Setup
import os
import time
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import onnxruntime as ort

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using compute device: {device}")

In [ ]:
# 3. Workload Data Generator
def generate_trace(n_points=8352, seed=42, base_load=0.25, n_spikes=15):
    np.random.seed(seed)
    t = np.arange(n_points)
    # 24-hour diurnal cycle (8352 points / 29 days = ~288 points/day)
    diurnal = 0.15 * np.sin(2 * np.pi * t / 288 - np.pi / 2)
    noise = np.random.normal(0, 0.03, size=n_points)
    cpu = base_load + diurnal + noise
    
    # Burst spikes
    spikes = np.random.choice(range(50, n_points - 50), size=n_spikes, replace=False)
    for idx in spikes:
        dur = np.random.randint(4, 12)
        amp = np.random.uniform(0.2, 0.45)
        shape = amp * np.exp(-0.5 * ((np.arange(dur) - dur / 2) / (dur / 4)) ** 2)
        end = min(idx + dur, n_points)
        cpu[idx:end] += shape[:end-idx]
        
    return np.clip(cpu, 0.02, 0.98)

raw_trace = generate_trace()
plt.figure(figsize=(12, 4))
plt.plot(raw_trace[:500], label="CPU Utilization (First 500 Steps)", color="#1f77b4")
plt.title("Simulated Cluster CPU Workload Trace")
plt.xlabel("Step (300s)")
plt.ylabel("CPU Usage Rate")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# 4. Sliding Window Preprocessing (24 steps)
WINDOW_SIZE = 24
min_v, max_v = float(raw_trace.min()), float(raw_trace.max())
scaled = (raw_trace - min_v) / (max_v - min_v)

X, y = [], []
for i in range(len(scaled) - WINDOW_SIZE):
    X.append(scaled[i:i + WINDOW_SIZE])
    y.append(scaled[i + WINDOW_SIZE])

X = np.array(X, dtype=np.float32).reshape(-1, WINDOW_SIZE, 1)
y = np.array(y, dtype=np.float32).reshape(-1, 1)

split = int(0.8 * len(X))
X_train, y_train = X[:split], y[:split]
X_test, y_test = X[split:], y[split:]

train_loader = DataLoader(TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train)), batch_size=512, shuffle=True)
print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")

In [ ]:
# 5. Define GRU Model & Training Loop
class GRUModel(nn.Module):
    def __init__(self, input_dim=1, hidden_dim=50, output_dim=1):
        super().__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, batch_first=True)
        self.relu = nn.ReLU()
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        out, _ = self.gru(x)
        return self.fc(self.relu(out[:, -1, :]))

model = GRUModel().to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

EPOCHS = 40
start_t = time.time()
for ep in range(1, EPOCHS + 1):
    model.train()
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        loss = criterion(model(bx), by)
        loss.backward()
        optimizer.step()
    if ep % 10 == 0 or ep == 1:
        print(f"Epoch [{ep:02d}/{EPOCHS}] - Loss: {loss.item():.6f}")

print(f"Training finished in {time.time() - start_t:.2f}s")

In [ ]:
# 6. Export to ONNX Runtime for Low-Latency K8s Deployment
model.eval().to("cpu")
dummy = torch.randn(1, WINDOW_SIZE, 1)
onnx_path = "GRU_Model_24.onnx"

torch.onnx.export(
    model, dummy, onnx_path,
    input_names=["input_sequence"],
    output_names=["predicted_cpu"],
    dynamic_axes={"input_sequence": {0: "batch_size"}, "predicted_cpu": {0: "batch_size"}}
)
print(f"ONNX Model exported: {onnx_path} ({os.path.getsize(onnx_path)/1024:.1f} KB)")

# Save Scaler parameters
with open("scaler.json", "w") as f:
    json.dump({"min_val": min_v, "max_val": max_v, "window_size": WINDOW_SIZE}, f)
print("Saved scaler.json")

In [ ]:
# 7. Verification & Latency Benchmark with ONNX Runtime
sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
in_name = sess.get_inputs()[0].name
out_name = sess.get_outputs()[0].name

test_input = X_test[:100]
t0 = time.time()
preds = sess.run([out_name], {in_name: test_input})[0]
lat_ms = ((time.time() - t0) / len(test_input)) * 1000.0

mse = mean_squared_error(y_test[:100], preds)
print(f"Test MSE: {mse:.6f}")
print(f"Average per-step inference latency: {lat_ms:.3f} ms (Well within 65ms limit)")

# Plot Predictions vs Ground Truth
plt.figure(figsize=(12, 4))
plt.plot(y_test[:100], label="Ground Truth CPU", color="black", alpha=0.7)
plt.plot(preds, label="GRU Proactive Predicted CPU", color="red", linestyle="--")
plt.title("Proactive Workload Prediction vs Actual Load (100 Steps)")
plt.xlabel("Time Step")
plt.ylabel("Normalized CPU")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()